# # 03 · Normalization with uMAIA

Companion to the tutorial notebook [`03_normalization_umaia`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level1).

**Kernel:** `cajal-umaia` (not `cajal-lipidomics`!) · **Reads:** `02_annotated.h5ad` · **Writes:** `data/derived/03_normalized.h5ad`

Our control and pregnant sections were acquired on different days, on different slides. The instrument never gives the same number twice for the same amount of lipid, so before asking "what changed in pregnancy?" we have to make the two sections comparable. That's what **uMAIA** does, and we'll unroll its correction by hand so it's not a black box.

> **Important caveat.** With 1 control vs 1 pregnant section, batch correction is ill-defined: any shift between them could be technical (batch) *or* biological (pregnancy, or just two different mice). Properly this is done at atlas scale with several sections and animals per condition. We do it 1 vs 1 for teaching, and keep this limitation in mind for every result downstream.

**The plan**
1. The batch effect, in one picture
2. The key empirical fact: per-lipid histograms are bimodal in log space
3. The uMAIA model (rank-1 batch shift), and how it's fit
4. Running uMAIA for real, and measuring the improvement
5. Unrolling the correction: histogram matching by CDF and inverse CDF
6. A second, separate normalization: per-lipid 0 to 1 scaling
7. Saving

In [ ]:
# ---- setup for notebook 03: runs on the cajal-umaia kernel ---------------------------------

# uMAIA's model is written in JAX. JAX goes hunting for a GPU (and CUDA) if we let it, so we
# pin it to the CPU. This MUST happen before anything imports jax, because JAX reads the setting
# once at import time. That's why it's the very first thing in the notebook.
import os
os.environ["JAX_PLATFORMS"] = "cpu"    # must be set BEFORE jax is imported: forces CPU, so it runs on any laptop without a GPU

# Same path helper as every other notebook
from lipid_project import paths
paths.ensure_dirs()
paths.summary()          # check that "annotated" is listed under "stages built"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import stats                        # the normal CDF, for unrolling the correction
from scipy.interpolate import interp1d         # turns a tabulated CDF into an inverse CDF

import jax                                     # the numerical engine uMAIA runs on
import uMAIA                                   # the normalization package (cloned into external/uMAIA)

import cajal_lipidomics as cl
from cajal_lipidomics import analysis
from cajal_lipidomics.style import FS
cl.style.set_style()

SEED = 42    # uMAIA's fit is stochastic; a fixed seed makes reruns give identical numbers
np.random.seed(SEED)       # uMAIA picks its pixel subsample with np.random, so seed it for reproducibility

# Should say "cpu". "No module named 'uMAIA'" means the wrong kernel: pick cajal-umaia (top right).
print("ready. JAX on", jax.devices()[0].platform)

In [ ]:
adata = ad.read_h5ad(paths.stage("annotated"))    # the output of notebook 02
X = np.asarray(adata.X, dtype=np.float32)                 # pixels x lipids, still the RAW scale
cond = adata.obs["Condition"].to_numpy()    # 'naive' or 'pregnant' per pixel, as a numpy array for fast masking

print(f"{adata.n_obs:,} pixels x {adata.n_vars} lipids")
print("pixels per condition:", adata.obs["Condition"].value_counts().to_dict())
print(f"raw X runs from {X.min():.3f} to {X.max():.3f}")

# `X` runs 0 to 1 because METASPACE serves each ion image as an 8-bit image scaled into [0, 1] per image. That's our starting scale.

## 1. The batch effect, in one picture

Between two acquisitions a lot drifts that has nothing to do with biology: how the matrix crystallized, the laser energy, the detector gain, how the slide was handled. Each nudges a whole section's intensities up or down. That's the **batch effect**.

Our saving grace: both sections are cut at the same coronal plane, so the **anatomy is matched**. Let's check that by painting each section with its Allen region colours.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, c in zip(axes, ["naive", "pregnant"]):
    m = cond == c
    ax.scatter(adata.obs.loc[m, "zccf"], -adata.obs.loc[m, "yccf"], c=adata.obs.loc[m, "allencolor"],
               s=2, rasterized=True)
    cl.style.spatial_axes(ax)
    ax.set_title(f"{c}  ({m.sum():,} pixels)", fontsize=FS["m"])
fig.suptitle("the two coronal sections, painted by Allen region", fontsize=FS["l"])
plt.tight_layout()
plt.show()

# Same plane, same anatomy, different mouse and different acquisition. Normalization's job is to remove the blanket technical difference while disturbing real, region-specific biology as little as possible.

## 2. The key fact: per-lipid histograms are bimodal in log space

Take one lipid, collect its intensity over all pixels of one section, and plot the histogram **in log space**. Usually you see two humps:
- a low **background** mode: pixels where the lipid is absent, so we measure matrix noise. It sits at a roughly fixed level.
- a higher **foreground** mode: pixels where the lipid is really there.

Log space matters because intensities span orders of magnitude; on a linear axis everything piles up near zero. We use `HexCer 42:2` (a myelin sphingolipid) if our annotation found it, otherwise the most myelin-like lipid we have.

In [ ]:
EPS = 2e-4                                                 # log offset, so log(0) doesn't blow up (uMAIA's value)
names = adata.var["lipid"].astype(str).to_list()    # readable lipid names, one per column


def first_index(prefixes):
    """Column index of the first lipid whose name starts with any of `prefixes`, else None."""
    for i, n in enumerate(names):
        if n.startswith(prefixes):
            return i
    return None


j = first_index(("HexCer 42:2",))    # the myelin lipid is the clearest bimodal example; the next lines fall back if it's absent
if j is None:
    j = first_index(("HexCer", "SM", "Cer"))               # any sphingolipid will do
if j is None:
    j = int(np.argmax(X.mean(0)))                          # last resort: the brightest ion
lipid = names[j]

vals = X[cond == "naive", j]    # this lipid's raw values on the control section
log_vals = np.log(np.clip(vals, 0, None) + EPS)    # clip kills tiny negatives, +EPS avoids log(0)
floor = np.log(EPS) + 0.3                                  # anything below this is at the background floor

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(vals, bins=80, color="steelblue")
axes[0].set_title("linear axis: the structure is invisible", fontsize=FS["m"])
axes[0].set_xlabel(f"{lipid} raw intensity"); axes[0].set_ylabel("pixels")
axes[1].hist(log_vals, bins=80, color="steelblue")
axes[1].set_title("log axis: two modes appear", fontsize=FS["m"])
axes[1].set_xlabel(f"log({lipid} + eps)")
plt.tight_layout()
plt.show()
print(f"{lipid}: {np.mean(log_vals < floor):.0%} of control pixels sit at the background floor")

# The crucial asymmetry (shown in the uMAIA paper): **the batch effect acts mostly on the foreground**. The background is matrix noise and stays put between slides, a stable **anchor**. So the correction shouldn't "shift everything by a constant". It should hold the anchor and slide the foreground back into register. **Anchor low, drift high.**

Let's overlay the same lipid from both sections. The dashed lines are each section's foreground median.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for c, colour in [("naive", "steelblue"), ("pregnant", "darkorange")]:
    lv = np.log(np.clip(X[cond == c, j], 0, None) + EPS)    # same log transform, per section
    ax.hist(lv, bins=80, density=True, histtype="stepfilled", alpha=0.45, color=colour, label=c)
    ax.axvline(np.median(lv[lv > floor]), color=colour, ls="--", lw=1)   # foreground median
ax.set_xlabel(f"log({lipid} + eps)"); ax.set_ylabel("density")
ax.set_title("same lipid, two sections: do the foreground modes line up?", fontsize=FS["m"])
ax.legend()
plt.tight_layout()
plt.show()

# ## 3. The uMAIA model

For every (lipid, section) pair, uMAIA fits a **two-Gaussian mixture in log space**: one Gaussian for background, one for foreground. The heart is how the foreground mean is built, for lipid *c* in section *a*:

$$\mu^{fg}_{ac} \;=\; \underbrace{\text{locs}_{c}}_{\text{background anchor (per lipid)}} \;+\; \underbrace{\gamma_a \cdot \lambda_c}_{\text{batch shift}} \;+\; \underbrace{\delta_c}_{\text{biological gap}} \;(+\ \text{small slack } \epsilon_{ac})$$

- `locs_c` is the background anchor, **one per lipid**, shared across sections.
- `γ_a` is one number per **slide**: how strong its drift is.
- `λ_c` is one number per **lipid**: how sensitive it is to drift.
- `δ_c` is the real fg-bg gap for that lipid, the same on every slide. It is **not** removed.

The batch shift is **rank-1** (slide factor × lipid factor). The paper justifies that empirically: an SVD of measured technical errors is dominated by its first component. Normalization removes `γ_a·λ_c` (and the slack) and keeps the anchor and `δ_c`.

**How it's fit.** Stochastic variational inference with a point-mass (`AutoDelta`) guide. In plain words, a **MAP point estimate**: find the single most probable parameter set by gradient descent (Adam), with the priors acting as gentle regularizers. No MCMC, no posterior samples.

**Three calls:**
1. `uMAIA.norm.initialize(x, mask)`: fit 1- and 2-component GMMs per molecule, keep the better (BIC), to get starting values
2. `uMAIA.norm.normalize(...)`: the MAP fit (the only slow bit; it subsamples ~2,500 pixels per section)
3. `uMAIA.norm.transform(x, mask, svi)`: apply the histogram-matching correction

# ### Building uMAIA's input

uMAIA normalizes each molecule *across* sections, so it wants a 3D tensor `(N_pixels, S_sections, V_molecules)` of log intensities, plus a boolean `mask` marking real pixels (the sections have different pixel counts, so the shorter one is padded with zeros that the mask switches off).

In [ ]:
sections = ["naive", "pregnant"]                             # section axis order: 0 = naive, 1 = pregnant
counts = [int((cond == s).sum()) for s in sections]          # real pixels in each section
N, S, V = max(counts), len(sections), X.shape[1]    # uMAIA wants a dense (pixels, sections, molecules) block, so N = the bigger section

x = np.zeros((N, S, V), np.float32)                          # padded log-intensity tensor
mask = np.zeros((N, S, V), bool)                             # True where a pixel actually exists
for s, name in enumerate(sections):
    xs = np.log(np.clip(X[cond == name], 0, None) + EPS)     # this section's log intensities
    x[: len(xs), s, :] = xs    # the smaller section is padded at the bottom...
    mask[: len(xs), s, :] = True    # ...and the mask marks which rows are real pixels vs padding
print(f"tensor x {x.shape} = (pixels, sections, molecules); real pixels per section {counts}")

# ### Running the fit

The progress bar shows the loss falling: that's the optimizer walking the parameters toward their most probable values. A couple of minutes on a laptop CPU.

Note `covariate_vector` stays at its default `None`: we do **not** tell uMAIA which section is pregnant. If we did, it would treat "pregnant" as a known group and could absorb the very biology we want to test later.

In [ ]:
init = uMAIA.norm.initialize(x, mask, subsample=True)                 # 1) GMM starting point per molecule
svi = uMAIA.norm.normalize(x, mask, init_state=init, subsample=True,  # 2) MAP fit via SVI
                           num_steps=2000, seed=SEED)
x_maia = np.asarray(uMAIA.norm.transform(x, mask, svi))              # 3) the correction, same shape as x
print("normalized tensor:", x_maia.shape)

# ### Did it help? Before vs after, per molecule

Top row: raw log histograms of the two sections. Bottom row: after uMAIA. We show four lipids, preferring ones from different classes.

In [ ]:
# Pick up to 4 named lipids from different classes (first of each class), to show variety
shown, seen = [], set()    # pick 4 lipids from 4 DIFFERENT classes for the before/after plot
for i, n in enumerate(names):
    cls = n.split(" ")[0]    # class = text before the first space
    if not n.startswith("ion_") and cls not in seen:
        shown.append(i); seen.add(cls)
    if len(shown) == 4:
        break

fig, axes = plt.subplots(2, len(shown), figsize=(3.5 * len(shown), 5.6), sharex="col")
for col, v in enumerate(shown):
    for s, colour in enumerate(["steelblue", "darkorange"]):
        m = mask[:, s, v]    # only real pixels; padding would add a fake spike at 0
        axes[0, col].hist(x[m, s, v], bins=60, density=True, alpha=0.5, color=colour, label=sections[s])
        axes[1, col].hist(x_maia[m, s, v], bins=60, density=True, alpha=0.5, color=colour, label=sections[s])
    axes[0, col].set_title(f"{names[v]}\nraw", fontsize=FS["s"])
    axes[1, col].set_title("uMAIA-normalized", fontsize=FS["s"])
    for r in (0, 1):
        axes[r, col].set_yticks([])
axes[0, 0].legend()
fig.suptitle("uMAIA slides each molecule's per-section distributions into register", fontsize=FS["l"])
plt.tight_layout()
plt.show()

# One number for all molecules: the gap between the two sections at the **90th percentile** (a robust stand-in for where the foreground sits), taking the median over molecules, before and after.

In [ ]:
def median_90pct_gap(arr):
    """Median over molecules of |q90(section 0) - q90(section 1)|, on real pixels only."""
    gaps = [abs(np.quantile(arr[mask[:, 0, v], 0, v], 0.9) - np.quantile(arr[mask[:, 1, v], 1, v], 0.9))
            for v in range(arr.shape[2])]
    return float(np.median(gaps))


gap_raw, gap_norm = median_90pct_gap(x), median_90pct_gap(x_maia)    # a single number for 'how far apart are the sections', before vs after
print("median cross-section gap at the 90th percentile (log units)")
print(f"  raw         : {gap_raw:.3f}")
print(f"  uMAIA       : {gap_norm:.3f}")
print(f"  shrinkage   : {100 * (1 - gap_norm / gap_raw):.0f}%")

# The gap shrinks but doesn't go to zero, and it shouldn't. With two sections uMAIA can't fully tell a real control-vs-pregnant difference from a batch effect, so it leaves some biology in place instead of over-correcting. (The exact numbers wobble a little between runs, because of the pixel subsample.)

### Back to the native scale, in AnnData row order

`x_maia` is in log space and padded. We undo the log with `exp` and stack the real pixels back in the same order as `adata` (all naive rows, then all pregnant rows, which is how notebook 01 stacked them).

In [ ]:
# Check the row order assumption before relying on it
assert (cond[: counts[0]] == "naive").all() and (cond[counts[0]:] == "pregnant").all(), \
    "expected all naive rows first, then all pregnant rows"

umaia_native = np.vstack([np.exp(x_maia[: counts[0], 0, :]),     # naive pixels (section axis 0)
                          np.exp(x_maia[: counts[1], 1, :])]     # pregnant pixels (section axis 1)
                         ).astype(np.float32)
assert umaia_native.shape == X.shape    # back to the flat (pixels, lipids) shape AnnData expects
print("uMAIA values on the native scale:", umaia_native.shape,
      f"range {umaia_native.min():.2e} to {umaia_native.max():.2f}")

# ### A quick biological check: the sphingolipid signal in white matter

Sphingolipids (HexCer, Cer, SM) are concentrated in **myelin**, so they light up white matter. We build a simple **composite score**: z-score each sphingolipid over pixels (so each counts equally), then average. White matter is defined the Allen atlas's own way: every fibre-tract region is painted with the colour `#cccccc` (notebook 04 explains this).

In [ ]:
sph_cols = [i for i, n in enumerate(names) if n.startswith(("HexCer", "Cer", "SM"))]    # sphingolipid columns: the planted biology to check survives normalization
wm = adata.obs["allencolor"].astype(str).str.lower().eq("#cccccc").to_numpy()
print(f"{len(sph_cols)} sphingolipid columns | {wm.sum():,} white-matter pixels ({wm.mean():.0%} of all)")


def sphingo_score(arr):
    """Mean z-scored sphingolipid intensity per pixel."""
    sub = arr[:, sph_cols]    # just the sphingolipids
    z = (sub - sub.mean(0)) / (sub.std(0) + 1e-9)        # z-score each column: mean 0, sd 1
    return z.mean(1)


# normalization should keep (not erase) a real white-matter difference between sections
for label, arr in [("raw", X), ("uMAIA", umaia_native)]:
    sc = sphingo_score(arr)
    nv, pg = sc[(cond == "naive") & wm].mean(), sc[(cond == "pregnant") & wm].mean()
    print(f"{label:6s}: white-matter sphingolipid z   naive {nv:+.3f}   pregnant {pg:+.3f}")

score = sphingo_score(umaia_native)    # one score per pixel to map
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, c in zip(axes, sections):
    m = cond == c
    sc_ = ax.scatter(adata.obs.loc[m, "zccf"], -adata.obs.loc[m, "yccf"], c=score[m], cmap="plasma",
                     vmin=np.quantile(score, 0.02), vmax=np.quantile(score, 0.98), s=2, rasterized=True)
    cl.style.spatial_axes(ax)
    ax.set_title(f"{c}: sphingolipid score (uMAIA)", fontsize=FS["m"])
cl.style.lightweight_colorbar(sc_, list(axes), label="mean z-scored sphingolipid")
plt.show()

# ## 5. Unrolling the correction: histogram matching

The `transform` call is about twenty lines of NumPy/SciPy. Suppose a lipid's log intensities in one section have CDF $G$ (value → quantile) and there's a shared **reference** distribution with CDF $F$. The whole correction is

$$x \;\longmapsto\; F^{-1}\big(G(x)\big)$$

Find each value's quantile in its own section, then read off the value at that same quantile in the reference. Because it preserves quantiles it is **monotone**: pixels never swap order within a section, the axis just gets restretched. And since $G$ and $F$ are two-Gaussian mixtures, the stretch can differ for background and foreground, which is how it holds the anchor while sliding the foreground.

In [ ]:
def cdf_mixture(x, mu1, mu2, sigma1, sigma2):
    """Equal-weight two-Gaussian mixture CDF: the average of two normal CDFs.
    (mu1, sigma1) = background mode, (mu2, sigma2) = foreground mode."""
    return (stats.norm.cdf(x, mu1, sigma1) + stats.norm.cdf(x, mu2, sigma2)) / 2.0


def make_inv_cdf(mu1, mu2, sigma1, sigma2, resolution=2000, rel_range=6):
    """F^-1 for a reference mixture: tabulate its CDF on a fine grid, then interpolate backwards.

    interp1d(cdf_values, grid) looks reversed on purpose: giving it quantiles as the x-axis and
    values as the y-axis yields a function quantile -> value, which is exactly the inverse CDF."""
    lo = min(mu1, mu2) - rel_range * max(sigma1, sigma2)
    hi = max(mu1, mu2) + rel_range * max(sigma1, sigma2)
    grid = np.linspace(lo, hi, resolution)
    return interp1d(cdf_mixture(grid, mu1, mu2, sigma1, sigma2), grid,
                    bounds_error=False, fill_value="extrapolate")


def transform_byicdf(x, mu1x, mu2x, sigma1x, sigma2x, icdf_ref):
    """The F^-1(G(x)) map: quantile under this section's mixture, value under the reference."""
    return icdf_ref(cdf_mixture(x, mu1x, mu2x, sigma1x, sigma2x))


print("built cdf_mixture, make_inv_cdf, transform_byicdf (same idea as uMAIA/normalization/_transform.py)")

# ### A controlled demo with a known answer

Two simulated sections of one lipid, each a background hump plus a foreground hump. Section B's foreground got shifted right and widened by a fake batch effect. We build a shared reference as the **mean of the two sections' parameters** (what uMAIA does), push both through $F^{-1}(G(x))$, and watch the foregrounds line up.

In [ ]:
n = 8000    # pixels per toy section
A_bg, A_fg, sA_bg, sA_fg = -8.0, -5.0, 0.30, 0.55        # section A: well behaved
B_bg, B_fg, sB_bg, sB_fg = -8.0, -4.0, 0.30, 0.80        # section B: same background, foreground drifted + wider


def sample_mixture(mu_bg, mu_fg, s_bg, s_fg, w_fg=0.55, seed=0):
    """Draw n pixels: each is foreground with probability w_fg, else background."""
    r = np.random.default_rng(seed)
    is_fg = r.random(n) < w_fg    # coin flip per pixel: foreground or background
    return np.where(is_fg, r.normal(mu_fg, s_fg, n), r.normal(mu_bg, s_bg, n))


A = sample_mixture(A_bg, A_fg, sA_bg, sA_fg, seed=1)
B = sample_mixture(B_bg, B_fg, sB_bg, sB_fg, seed=2)

# Shared reference = the average of the two sections' mixture parameters
# the reference = the average of both sections' mixtures, like uMAIA does
ref = dict(mu1=(A_bg + B_bg) / 2, mu2=(A_fg + B_fg) / 2, sigma1=(sA_bg + sB_bg) / 2, sigma2=(sA_fg + sB_fg) / 2)
icdf_ref = make_inv_cdf(**ref)    # ** unpacks the dict into keyword arguments
A_corr = transform_byicdf(A, A_bg, A_fg, sA_bg, sA_fg, icdf_ref)    # each section is mapped with ITS OWN fitted mixture onto the shared reference
B_corr = transform_byicdf(B, B_bg, B_fg, sB_bg, sB_fg, icdf_ref)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True, sharey=True)
for ax, (a, b), title in [(axes[0], (A, B), "before: foregrounds misaligned"),
                          (axes[1], (A_corr, B_corr), "after: both snapped onto the reference")]:
    ax.hist(a, bins=80, density=True, alpha=0.5, color="steelblue", label="A")
    ax.hist(b, bins=80, density=True, alpha=0.5, color="darkorange", label="B")
    ax.axvline(ref["mu2"], color="k", ls="--", lw=1)    # dashed line = where the reference foreground sits
    ax.set_title(title, fontsize=FS["m"]); ax.set_xlabel("log-intensity"); ax.legend()
plt.tight_layout()
plt.show()

fg = lambda arr: np.median(arr[arr > -6.5])                 # median of the foreground part
print(f"before: A fg median {fg(A):.2f}, B fg median {fg(B):.2f}")
print(f"after : A fg median {fg(A_corr):.2f}, B fg median {fg(B_corr):.2f}  (reference fg = {ref['mu2']:.2f})")

# Both foregrounds land on the reference, the background barely moves, and each pixel kept its rank. No constant subtracted, no scalar divided: quantiles matched through fitted CDFs.

With only two sections, the reference is just the midpoint, so if one slide were a technical outlier it would drag the reference halfway toward itself. That's another reason real studies use many sections.

# ## 6. A second, separate normalization: per-lipid 0 to 1

uMAIA makes the **same lipid** comparable **across sections**. A different problem: **different lipids** aren't comparable to each other. MALDI is only semi-quantitative, since each molecule ionizes with a different efficiency. A bright ion isn't necessarily an abundant lipid.

Fix: clip each lipid's extreme 0.5% tails and rescale it to [0, 1], so 0.7 means "high *for this lipid*". This is what every cross-lipid step later runs on (the embedding in notebook 05). We unroll it and check it against the helper `cl.analysis.min01_per_lipid`.

In [ ]:
lo_q, hi_q = 0.005, 0.995    # robust min/max: ignore the extreme 0.5% on each side so one hot pixel can't squash the scale
ql = np.quantile(umaia_native, lo_q, axis=0)          # per-lipid low clip (one value per column)
qh = np.quantile(umaia_native, hi_q, axis=0)          # per-lipid high clip
span = np.where(qh > ql, qh - ql, 1.0)                 # guard: a flat lipid would divide by zero
X01 = np.clip((umaia_native - ql) / span, 0, 1)    # rescale every lipid so its robust range maps to [0, 1]

X01_helper = analysis.min01_per_lipid(umaia_native, lo=lo_q, hi=hi_q)    # the tutorial's helper, to check my hand-rolled version
print("max |hand-rolled - helper| =", float(np.max(np.abs(X01 - X01_helper))))

# See the scales collapse onto [0, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for c in shown[:3]:
    axes[0].hist(umaia_native[:, c], bins=80, histtype="step", lw=1.4, label=names[c])
    axes[1].hist(X01[:, c], bins=80, histtype="step", lw=1.4, label=names[c])
axes[0].set_title("before: each lipid on its own scale", fontsize=FS["m"]); axes[0].set_xlabel("uMAIA intensity")
axes[1].set_title("after: every lipid on a common [0, 1]", fontsize=FS["m"]); axes[1].set_xlabel("per-lipid 0-1 value")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

# ## 7. Save

We keep raw `X` untouched and store the uMAIA values as a **layer** on the native scale. Every later notebook reads `layers["umaia"]`. (The 0 to 1 scaling is cheap and gets recomputed where it's needed, in notebook 05.)

In [ ]:
adata.layers["umaia"] = umaia_native    # raw X stays in .X; the normalized values go in a layer next to it
adata.uns["umaia"] = {"num_steps": 2000, "seed": SEED, "eps": EPS,
                      "gap90_raw": gap_raw, "gap90_umaia": gap_norm}   # a record of what we did
adata.write_h5ad(paths.stage("normalized"))        # -> data/derived/03_normalized.h5ad
print("saved", paths.stage("normalized"), "| layers:", list(adata.layers.keys()))

# ## Recap

- The **batch effect** shifts whole acquisitions for technical reasons. With 1 vs 1 sections it is tangled with biology.
- MALDI log histograms are **bimodal**: a stable background anchor and a drifting foreground.
- uMAIA's foreground mean has a **rank-1 batch term** `γ_a·λ_c`, fit by **MAP via SVI**, and the correction is quantile matching $F^{-1}(G(x))$.
- **uMAIA** makes one lipid comparable across sections; **min01** makes different lipids comparable to each other.
- **Differential tests always run on the uMAIA layer, never on Harmony-corrected embeddings** (more in 05).
- Saved **`data/derived/03_normalized.h5ad`** with `layers["umaia"]`.

**Next (04):** anatomy, registration and the Allen atlas.